# Qualified process-to-engineering reference facility

This notebook executes the connected inlet separator → compressor → cooler → export-line reference facility, including anti-surge recycle, LCV/PCV, ESDVs, PSV, BDV, flare coupling, ten design cases, iterative physical sizing, dynamic safe-state verification, DEXPI 2.0 and the coordinated package. All embedded evidence is synthetic regression evidence. Passing the workflow qualifies a controlled software pilot only; `fitnessForConstruction` remains false.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path(os.environ.get('NEQSIM_PROJECT_ROOT', Path.cwd())).resolve()
while not (ROOT / 'pom.xml').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'devtools'))
from neqsim_dev_setup import neqsim_init, neqsim_classes
ns = neqsim_classes(neqsim_init(project_root=ROOT, recompile=not (ROOT / 'target/classes').exists(), verbose=False))
JClass = ns.JClass
print('Loaded NeqSim from', ROOT)

## 1. Build and inspect the controlled facility

The Java factory supplies the complete topology and evidence contracts. A project implementation should replace the synthetic values, while preserving the fail-closed structure.

In [ ]:
Reference = JClass('neqsim.process.engineering.verticalslice.InletCompressionExportReferenceFacility')
Preflight = JClass('neqsim.process.engineering.verticalslice.ProductionVerticalSlicePreflight')
definition = Reference.build()
project = definition.getProject()
preflight = Preflight.assess(project, definition.getQualificationPolicy())
print('Cases:', len(project.getExecutableDesignCases()))
print('Ready for simulation:', preflight.isReadyForSimulation())
print('Blockers:', list(preflight.getBlockers()))
assert preflight.isReadyForSimulation()

## 2. Run the closed design loop and compile the package

Strict execution refuses incomplete topology, cases, compressor maps, dynamic scenarios, safety studies, standards or evidence before starting thermodynamics.

In [ ]:
Simulator = JClass('neqsim.process.engineering.verticalslice.ProductionVerticalSliceSimulator')
Paths = JClass('java.nio.file.Paths')
package_dir = ROOT / 'build' / 'qualified-engineering-reference-facility'
package_dir.mkdir(parents=True, exist_ok=True)
result = Simulator.runStrictAndCompile(project, definition.getAutoConfigurationPolicy(),
    definition.getQualificationPolicy(), 1, Paths.get(str(package_dir)), None)
loop = result.getSimulation().getEngineeringDesignLoopResult()
print('Design loop converged:', loop.isConverged(), loop.getTerminationReason())
print('Qualified for controlled pilot:', result.getQualification().isQualifiedForControlledPilot())
print('Failed gates:', list(result.getQualification().getFailedGates()))
print('DEXPI 2.0:', result.getCompilation().getDexpiResult().getDexpi20File())
assert loop.isConverged()
assert result.getQualification().isQualifiedForControlledPilot()

## 3. Review governing physical design variables

In [ ]:
import pandas as pd
state = loop.getState().getValues()
rows = []
for key in state.keySet():
    value = state.get(key)
    item = value.toMap()
    rows.append({'variable': str(key), 'value': item.get('value'), 'unit': str(item.get('unit')),
                 'governing_case': str(item.get('governingCaseId')), 'module': str(item.get('sourceModule'))})
design_table = pd.DataFrame(rows).sort_values('variable')
design_table[design_table.variable.str.contains('insideDiameter|driverRatedPower|selectedCv|selectedOrificeArea|preliminaryHeatTransferArea', regex=True)]

## 4. Visualize the case envelope

The chart shows the compressor shaft-power envelope used to select driver rating.

In [ ]:
import matplotlib.pyplot as plt
case_results = result.getSimulation().getCaseRunReport().getEnvelope().getCaseResults()
case_ids = [str(item.getDesignCase().getId()) for item in case_results]
powers = [item.getValues().get('EXPORT-COMP.power') for item in case_results]
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(case_ids, powers, color='#2378b5')
ax.set_ylabel('Compressor shaft power [kW]')
ax.set_xlabel('Controlled design case')
ax.set_title('Qualified reference-facility compressor envelope')
ax.grid(axis='y', alpha=0.3)
ax.tick_params(axis='x', rotation=35)
fig.tight_layout()
plt.show()

**Discussion.** The governing bar identifies the case that controls preliminary driver selection. Changes in feed rate, composition or discharge pressure change the execution fingerprint and force recalculation. Engineering implication: use the envelope for pre-FEED screening, then replace the synthetic map and driver candidates with vendor-supported data.

## 5. Verify package and approval boundary

In [ ]:
required = ['plant.dexpi.xml', 'plant.dexpi20.xml', 'equipment-register.json', 'line-register.json',
            'instrument-register.json', 'valve-list.json', 'io-list.json', 'alarm-trip-schedule.json',
            'flare-blowdown-report.json', 'materials-selection-report.json',
            'engineering-vertical-slice-execution-manifest.json']
pd.DataFrame([{'artifact': name, 'exists': (package_dir / name).exists()} for name in required])
print('fitnessForConstruction =', result.toMap().get('fitnessForConstruction'))
print('engineeringApprovalRequired =', result.toMap().get('engineeringApprovalRequired'))

A successful run does not approve HAZOP/LOPA decisions, SIL, vendor performance, final metallurgy, piping stress, certified PSV capacity or construction. These remain accountable project approvals and independent qualification evidence.